In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer

import torch
import torch_directml
from torch.utils.data import Dataset, DataLoader
from torch import nn

In [2]:
df = pd.read_csv('emotion/split/train.csv')
test_df = pd.read_csv('emotion/split/test.csv')
validation_df = pd.read_csv('emotion/split/validation.csv')

In [3]:
df

,text,label
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,3
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,3
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,1
15998,i feel like this was such a rude comment and i...,3


In [4]:
test_df

,text,label
0,im feeling rather rotten so im not very ambiti...,0
1,im updating my blog because i feel shitty,0
2,i never make her separate from me because i do...,0
3,i left with my bouquet of red and yellow tulip...,1
4,i was feeling a little vain when i did this one,0
...,...,...
1995,i just keep feeling like someone is being unki...,3
1996,im feeling a little cranky negative after this...,3
1997,i feel that i am useful to my people and that ...,1
1998,im feeling more comfortable with derby i feel ...,1


In [5]:
validation_df

,text,label
0,im feeling quite sad and sorry for myself but ...,0
1,i feel like i am still looking at a blank canv...,0
2,i feel like a faithful servant,2
3,i am just feeling cranky and blue,3
4,i can have for a treat or if i am feeling festive,1
...,...,...
1995,im having ssa examination tomorrow in the morn...,0
1996,i constantly worry about their fight against n...,1
1997,i feel its important to share this info for th...,1
1998,i truly feel that if you are passionate enough...,1


In [ ]:
def make_loercase(text):
    return text.lower()

def remove_link(text):
    return re.sub(r'https?://\S+|www\.\S+', '', text)

def remove_number(text):
    return re.sub(r'\d+', '', text)

def remove_punctuation(text):
    return re.sub(r'[^\w\s]', '', text)


In [7]:
df['text'] = df['text'].apply(word_tokenize) #test dataset


In [8]:
test_df['text'] = test_df['text'].apply(word_tokenize) #test dataset

In [9]:
validation_df['text'] = validation_df['text'].apply(word_tokenize) #validation dataset

wort to vocab

In [10]:
row_data_trin = df['text']
row_data_test = test_df['text']
row_data_valu = validation_df['text']

row_data = pd.concat([row_data_trin, row_data_test, row_data_valu], ignore_index=True)
# type(row_data),len(row_data), len(row_data_trin), len(row_data_test), len(row_data_valu)

vocab = [word for token in row_data for word in token]
# set(vocab)
vocab = {word: i+1 for i, word in(enumerate(set(vocab)))}
vocab.update({'<unknown>': 0})
len(vocab)

17095

maek vetor

In [11]:
lable_map ={
    0: "sadness",
    1: "joy",
    2: "love",
    3: "anger",
    4: "fear",
    5: "surprise"
}

In [12]:
Tf_idf_Vectorizer = TfidfVectorizer()

In [13]:
Tf_idf_Vector = Tf_idf_Vectorizer.fit_transform(df['text'].apply(lambda x: ' '.join(x)))
Tf_idf_Vector = torch.tensor(Tf_idf_Vector.toarray(), dtype=torch.float32) #train staset

In [14]:
test_idf_vector = Tf_idf_Vectorizer.transform(test_df['text'].apply(lambda x: ' '.join(x))) #train dataset
test_idf_vector = torch.tensor(test_idf_vector.toarray(), dtype=torch.float32)

In [15]:
validation_idf_vector = Tf_idf_Vectorizer.transform(validation_df['text'].apply(lambda x: ' '.join(x))) #validationn dataset
validation_idf_vector = torch.tensor(validation_idf_vector.toarray(), dtype=torch.float32)

In [16]:
# make dateset and dataloader

class dataset(Dataset):
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels
    def __len__(self):
        return len(self.data)
    def __getitem__(self, key):
        return self.data[key], self.labels[key]

In [17]:
type(Tf_idf_Vector), torch.tensor(df['label'])

(torch.Tensor, tensor([0, 0, 3,  ..., 1, 3, 0]))

In [18]:
train_dataset = dataset(Tf_idf_Vector, torch.tensor(df['label'].values)) #test dataset
# train_data_loader = DataLoader(train_dataset,batch_size=30,shuffle=True)
# len(train_data_loader)

In [19]:
test_dataset = dataset(test_idf_vector, torch.tensor(test_df['label'].values)) #test dataset
# test_data_loader = DataLoader(test_dataset,batch_size=30,shuffle=False)

In [20]:
validation_dataset = dataset(validation_idf_vector, torch.tensor(validation_df['label'].values)) #validation dataset
# validation_data_loader = DataLoader(validation_dataset,batch_size=30,shuffle=False)

make the model

In [21]:
# class AnnModel(nn.Module):
#     def __init__(self, input_number):
#         super(AnnModel, self).__init__()
#         self.model = nn.Sequential(
#             nn.Linear(input_number, 128),
#             nn.BatchNorm1d(128),
#             nn.ReLU(),
#             nn.Dropout(p=0.3),
#             nn.Linear(128, 64),
#             nn.BatchNorm1d(64),
#             nn.ReLU(),
#             nn.Dropout(p=0.3),
#             nn.Linear(64, 6)
#         )
#     def forward(self, x):
#         return self.model(x)

In [22]:
# learning_rate = 0.001
# num_epochs = 100

In [23]:
# device = torch_directml.device()

# # model = AnnModel(Tf_idf_Vector.shape[1]).to(device) # instatiate the model
# model = AnnModel(Tf_idf_Vector.shape[1])# instatiate the model

# loss_function = nn.CrossEntropyLoss() # define the loss function

# optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)# define the optimizer

tarin the model

In [24]:
# # train pipline on gpu
# for epoch in range(num_epochs):
#     total_loss = 0
#     for text,label in train_data_loader:
#         text,label = text.to(device), label.to(device)
#         output = model(text) # forward pass
#         loss = loss_function(output, label) # calculate the loss
#         optimizer.zero_grad() # zero the gradients
#         loss.backward() # backpropagation
#         optimizer.step()    # update the weights

#         total_loss += loss.item()
#     print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_data_loader):.4f}')


In [25]:
# # train pipline
# for epoch in range(num_epochs):
#     total_loss = 0
#     for text,label in train_data_loader:
#         output = model(text) # forward pass
#         loss = loss_function(output, label) # calculate the loss
#         optimizer.zero_grad() # zero the gradients
#         loss.backward() # backpropagation
#         optimizer.step()    # update the weights

#         total_loss += loss.item()
#     print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_data_loader):.4f}')

In [26]:
# Epoch [1/100], Loss: 1.5485
# Epoch [2/100], Loss: 1.2839
# Epoch [3/100], Loss: 1.1741
# Epoch [4/100], Loss: 1.1257
# Epoch [5/100], Loss: 1.1038
# Epoch [6/100], Loss: 1.0963
# Epoch [7/100], Loss: 1.0928
# Epoch [8/100], Loss: 1.0910
# Epoch [9/100], Loss: 1.0880
# Epoch [10/100], Loss: 1.0761
# Epoch [11/100], Loss: 1.0645
# Epoch [12/100], Loss: 1.0594
# Epoch [13/100], Loss: 1.0582
# Epoch [14/100], Loss: 1.0574
# Epoch [15/100], Loss: 1.0567
# Epoch [16/100], Loss: 1.0559
# Epoch [17/100], Loss: 1.0557
# Epoch [18/100], Loss: 1.0556
# Epoch [19/100], Loss: 1.0551
# Epoch [20/100], Loss: 1.0545
# Epoch [21/100], Loss: 1.0545
# Epoch [22/100], Loss: 1.0540
# Epoch [23/100], Loss: 1.0538
# Epoch [24/100], Loss: 1.0537
# Epoch [25/100], Loss: 1.0538
# ...
# Epoch [97/100], Loss: 1.0508
# Epoch [98/100], Loss: 1.0508
# Epoch [99/100], Loss: 1.0508
# Epoch [100/100], Loss: 1.0508

In [27]:
# model.eval()

In [28]:
# total = 0
# correct = 0
# for text, label in train_data_loader:
#     # text,label = text.to(device), label.to(device)
#     output = model(text)
#     _, predicted = torch.max(output.data, 1)
#     total += label.size(0)
#     correct += (predicted == label).sum().item()
#     print(f'Accuracy: {100 * correct / total:.2f}%')

In [29]:
# # test the model on test dataset
# with torch.no_grad():
#     total = 0
#     correct = 0
#     for text, label in test_data_loader:
#         # text,label = text.to(device), label.to(device)
#         # print(len(text))
#         output = model(text)
#         _, predicted = torch.max(output.data, 1)
#         total += label.size(0)
#         correct += (predicted == label).sum().item()
#     print(f'Test Accuracy: {100 * correct / total:.2f}%')

In [30]:
# # take user input
# user_input = input("Enter a text: ")
# # preprocess the user input
# user_input = user_input.lower()
# user_input = remove_link(user_input)
# user_input = remove_number(user_input)
# user_input = remove_punctuation(user_input)
# user_input = Tf_idf_Vectorizer.transform([user_input])
# user_input = torch.tensor(user_input.toarray(), dtype=torch.float32)
# model.eval()
# with torch.no_grad():
#     output = model(user_input)
#     _, predicted = torch.max(output.data, 1)
#     emotion = lable_map[predicted.item()]
#     print(f'Predicted Emotion: {emotion}')
    


In [31]:
import optuna
study = optuna.create_study(direction='maximize')

[I 2026-04-26 22:58:00,153] A new study created in memory with name: no-name-ffd4196e-95e5-4305-9e26-c4769310992c


In [32]:
class AnnModel(nn.Module):
    def __init__(self, input_number,output_number, number_of_hiden_layer, number_of_neuron,drope_out_rate):
        super(AnnModel, self).__init__()
        Layers = []
        for _ in range(number_of_hiden_layer):
            Layers.append(nn.Linear(input_number, number_of_neuron))
            Layers.append(nn.BatchNorm1d(number_of_neuron))
            Layers.append(nn.ReLU())
            Layers.append(nn.Dropout(p=drope_out_rate))
            input_number = number_of_neuron
        
        Layers.append(nn.Linear(number_of_neuron, output_number))
        self.model = nn.Sequential(*Layers)

        #model 
    def forward(self, x):
        return self.model(x)

In [33]:
def objective(trail):
    # Define the hyperparameters to tune
    number_of_hiden_layer  = trail.suggest_int('number_of_hiden_layer', 1, 3)
    number_of_neuron = trail.suggest_int('number_of_neuron', 32, 256) 
    number_of_epoch = trail.suggest_int('number_of_epoch', 10, 100)
    number_learning_rate = trail.suggest_float('learning_rate', 1e-5, 1e-2, log=True)
    drope_out_rate = trail.suggest_float('drope_out_rate', 0.1, 0.5)
    optimizer_name = trail.suggest_categorical('optimizer', ['Adam', 'SGD','RMSprop'])
    weight_decay = trail.suggest_float('weight_decay', 1e-5, 1e-3, log=True)
    batch_size = trail.suggest_int('batch_size', 32, 256)


    #paramitar intilize
    # learning_rate = 0.001
    # epoch = 100
    train_data_loader = DataLoader(train_dataset,batch_size=batch_size,shuffle=True)
    # test_data_loader = DataLoader(test_dataset,batch_size=30,shuffle=False)
    validation_data_loader = DataLoader(validation_dataset,batch_size=batch_size,shuffle=False)


    model = AnnModel(Tf_idf_Vector.shape[1], 6, number_of_hiden_layer, number_of_neuron, drope_out_rate)

    # optimizer
    loss_function = nn.CrossEntropyLoss() # define the loss function
    
    if optimizer_name == 'Adam':
        optimizer = torch.optim.Adam(model.parameters(), lr=number_learning_rate, weight_decay=weight_decay)# define the optimizer
    elif optimizer_name == 'SGD':
        optimizer = torch.optim.SGD(model.parameters(), lr=number_learning_rate, weight_decay=weight_decay)# define the optimizer
    else:
        optimizer = torch.optim.RMSprop(model.parameters(), lr=number_learning_rate, weight_decay=weight_decay)# define the optimizer


    # traing loop
    for epoch in range(number_of_epoch):
        for text,label in train_data_loader:
            output = model(text) # forward pass

            loss = loss_function(output, label) # calculate the loss

            optimizer.zero_grad() # zero the gradients

            loss.backward() # backpropagation

            optimizer.step()    # update the weights
    
    #model evalutions
    model.eval() 

    total = 0
    correct = 0
    for text, label in validation_data_loader:
        # text,label = text.to(device), label.to(device)

        output = model(text)

        _, predicted = torch.max(output.data, 1)

        total += label.size(0)

        correct += (predicted == label).sum().item()

    acuuresy  = 100 * correct / total
    return acuuresy

In [34]:
study.optimize(objective, n_trials=10)

[I 2026-04-26 23:03:57,013] Trial 0 finished with value: 88.7 and parameters: {'number_of_hiden_layer': 3, 'number_of_neuron': 47, 'number_of_epoch': 84, 'learning_rate': 0.00044152274529601145, 'drope_out_rate': 0.3530165456045472, 'optimizer': 'RMSprop', 'weight_decay': 0.0003690905549013187, 'batch_size': 102}. Best is trial 0 with value: 88.7.
[I 2026-04-26 23:09:02,083] Trial 1 finished with value: 87.95 and parameters: {'number_of_hiden_layer': 1, 'number_of_neuron': 183, 'number_of_epoch': 75, 'learning_rate': 0.0021066310630430615, 'drope_out_rate': 0.47214085123896044, 'optimizer': 'Adam', 'weight_decay': 4.696417312144453e-05, 'batch_size': 205}. Best is trial 0 with value: 88.7.
[I 2026-04-26 23:12:45,960] Trial 2 finished with value: 89.7 and parameters: {'number_of_hiden_layer': 1, 'number_of_neuron': 126, 'number_of_epoch': 69, 'learning_rate': 0.0030501016886206153, 'drope_out_rate': 0.3622548228835848, 'optimizer': 'SGD', 'weight_decay': 0.0006639858875567667, 'batch_si

In [35]:
study.best_value

89.7

In [36]:
study.best_params  

{'number_of_hiden_layer': 1,
 'number_of_neuron': 126,
 'number_of_epoch': 69,
 'learning_rate': 0.0030501016886206153,
 'drope_out_rate': 0.3622548228835848,
 'optimizer': 'SGD',
 'weight_decay': 0.0006639858875567667,
 'batch_size': 96}

87.4
{'number_of_hiden_layer': 2,
 'number_of_neuron': 50,
 'number_of_epoch': 69,
 'learning_rate': 0.00028102038216883563,
 'drope_out_rate': 0.3558712045121146,
 'optimizer': 'SGD',
 'weight_decay': 0.0008786781696825277,
 'batch_size': 69}

In [37]:
save model

The following commands were written to file `model.py`:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer

import torch
import torch_directml
from torch.utils.data import Dataset, DataLoader
from torch import nn
df = pd.read_csv('emotion/split/train.csv')
test_df = pd.read_csv('emotion/split/test.csv')
validation_df = pd.read_csv('emotion/split/validation.csv')
df
test_df
validation_df
def make_loercase(text):
    return text.lower()

def remove_link(text):
    return re.sub(r'https?://\S+|www\.\S+', '', text)

def remove_number(text):
    return re.sub(r'\d+', '', text)

def remove_punctuation(text):
    return re.sub(r'[^\w\s]', '', text)
df['text'] = df['text'].apply(word_tokenize) #test dataset
test_df['text'] = test_df['text'].apply(word_tokenize) #test dataset
validation_df['text'] = validation_df['text